# Sensibilidad a la temperatura · Zenteno G/F y S

Comparamos el G/F del notebook 16 y el S principal del 17 con sus parámetros guardados. Los dos escenarios de cada modelo conservan ecuaciones, condiciones iniciales, pulsos, muestras y parámetros de gas. LAB004–009 son calibración química, LAB019–021 son calibración de gas y LAB016–018 / LAB022–024 son holdout.

La alternativa toma la mediana de las lecturas de cada minuto cerrado y mantiene el último valor disponible durante los huecos. Si no hay un minuto cerrado al inicio, conserva el perfil original hasta la primera mediana disponible. Ese prefijo usa la preparación original y queda marcado en la figura de temperatura. La comparación es offline y ese prefijo no representa una reconstrucción causal.

Las figuras usan las mismas muestras y horizontes en ambas variantes. El RMSE excluye la muestra inicial. Para el CO₂ se conserva el soporte del notebook de origen, medianas horarias para LAB016–021 y medias de 30 minutos para LAB022–024. Los RMSE de esos dos soportes se leen por separado.

In [ ]:
from pathlib import Path
from types import FunctionType, SimpleNamespace
from unittest.mock import patch
import json
import os
import re
import sys

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/wine_fermentation').is_dir())
assert Path(sys.prefix).name.lower() == 'fermentation'
assert Path(os.environ['CONDA_PREFIX']).resolve() == Path(sys.prefix).resolve()
print(sys.executable)
directory = root / 'notebooks'

def source(stem):
    return nbformat.read(directory / (stem + '.source.ipynb'), as_version=4)

def cell(notebook, marker):
    matches = [c.source for c in notebook.cells if c.cell_type == 'code' and marker in c.source]
    assert len(matches) == 1, marker
    return matches[0]

def run(context, source_code):
    exec(compile(source_code, context['__name__'], 'exec'), context)

stems = {
    'G/F': '16_zenteno_recalibration_legacy_treatment_variable_solubility',
    'S': '17_zenteno_single_sugar_variable_solubility',
}
gf = source(stems['G/F'])
single = source(stems['S'])
# Se reutilizan las definiciones y la preparación de los notebooks originales.
# Los fragmentos terminan antes de cada llamada de ajuste y de las figuras originales.
ctx = {'__name__': stems['G/F'], 'display': lambda *args: None}
setup = cell(gf, 'INITIAL_VIABLE_MCELLS_ML')
setup = '\n'.join(line for line in setup.splitlines() if not line.startswith(('protected=', 'hashes=')))
run(ctx, setup)
run(ctx, cell(gf, 'def simulate(b,p):'))
run(ctx, cell(gf, 'Química congelada').split('traincurves=')[0])
gas_source = cell(gf, 'def calibrate_gas')
batch_setup = gas_source.split('PCO2_BAR=')[0]
first = batch_setup.index(' batches[lab]=b;f=simulate')
last = batch_setup.index(' obs[lab]=obs[lab].loc', first)
run(ctx, batch_setup[:first] + ' batches[lab]=b\n' + batch_setup[last:])
run(ctx, 'PCO2_BAR=' + gas_source.split('PCO2_BAR=', 1)[1].split('def calibrate_gas')[0])
s_context = ctx.copy()
s_context['__name__'] = stems['S']
s_source = cell(single, 'def simulate_s')
run(s_context, 'YXS=' + s_source.split('YXS=', 1)[1].split('support_s=')[0])
run(s_context, 'ps={' + s_source.split('\nps={', 1)[1].split('\nassert ps')[0])

contexts = {'G/F': ctx, 'S': s_context}
for model, notebook in [('G/F', gf), ('S', single)]:
    context = contexts[model]
    run(context, cell(notebook, 'def read_holdout_backup'))
    holdout_code = cell(notebook, 'def simulate_holdout')
    run(context, holdout_code.split('\nholdout_kinetic,')[0])
    context['predict_holdout'] = holdout_code[holdout_code.index('holdout_kinetic,'):]
    executed = nbformat.read(directory / (stems[model] + '.executed.ipynb'), as_version=4)
    saved = [line.removeprefix('ALL_GAS_FITS_JSON ') for c in executed.cells
             for output in c.get('outputs', []) for line in output.get('text', '').splitlines()
             if line.startswith('ALL_GAS_FITS_JSON ')]
    assert len(saved) == 1
    fit = json.loads(saved[0])[model]
    context['gas_parameters'] = (fit['kLa'], fit['gain'])
    np.testing.assert_allclose(context['gas_parameters'], context['HOLDOUT_GAS_PARAMS'], rtol=0, atol=1e-14)
    print(model, 'kLa y ganancia guardados:', context['gas_parameters'])

In [ ]:
kin = ctx['kin']
reference_temperature = kin.temperature_at
calibration = {b.batch: b for b in ctx['treated']}
historical = ctx['batches']
recent = ctx['holdout_batches']
all_batches = {**calibration, **historical, **recent}
roles = {lab: ('Calibración química' if lab in calibration else
               'Calibración gas' if lab in ctx['TRAIN'] else 'Holdout') for lab in all_batches}

def old_sensor(lab):
    clocks = pd.read_csv(root / 'data/compatibility/chemistry/clocks.csv').set_index('batch').t0
    paths = sorted(path for path in (root / 'data/original/natural/process_logs').rglob('*.csv')
                   if path.name.lower().startswith('temp') and
                   re.search(rf'lab0*{int(lab[3:])}(?:\D|$)', path.name, re.I))
    raw = pd.concat([pd.read_csv(path, usecols=['timestamp', 'T']) for path in paths])
    raw['timestamp'] = pd.to_datetime(raw.timestamp, errors='coerce')
    raw['T'] = pd.to_numeric(raw['T'], errors='coerce')
    raw = raw.dropna().sort_values('timestamp', kind='stable').drop_duplicates('timestamp')
    return raw.assign(t_h=(raw.timestamp - pd.Timestamp(clocks[lab])).dt.total_seconds() / 3600)

def measured_temperature(lab):
    if lab in calibration:
        return old_sensor(lab)
    if lab in historical:
        return ctx['v'].raw_temperature[lab]
    raw = ctx['holdout_temperature']
    return raw.loc[raw.lab.eq(lab)].rename(columns={'h': 't_h'})

profiles_1min = {}
coverage = []
for lab, batch in all_batches.items():
    raw = measured_temperature(lab).dropna(subset=['t_h', 'T'])
    minute = np.floor(raw.t_h.to_numpy() * 60 + 1e-10).astype(int)
    median = raw.groupby(minute).T.median()
    available = (median.index.to_numpy() + 1) / 60
    profiles_1min[lab] = (available, median.to_numpy())
    first, end = batch.time[[0, -1]]
    in_range = available[(available > first) & (available <= end)]
    assert len(in_range), lab
    wait = 0. if np.any(available <= first) else in_range[0] - first
    breaks = np.r_[first, in_range, end]
    coverage.append({'LAB': lab, 'Rol': roles[lab], 'Inicio [h]': first,
                     'Hasta [h]': end, 'Prefijo original [min]': 60 * wait,
                     'Mayor intervalo entre actualizaciones [min]': np.diff(breaks).max() * 60})

display(pd.DataFrame(coverage).set_index('LAB').round(2))

def causal_temperature(batch, t):
    available, values = profiles_1min[batch.batch]
    index = np.searchsorted(available, t + 1e-12, side='right') - 1
    return float(values[index]) if index >= 0 else reference_temperature(batch, t)

def temperature_figure(labs, title):
    fig, axes = plt.subplots(len(labs) // 3, 3, figsize=(15, 3.4 * len(labs) // 3))
    for ax, lab in zip(axes.flat, labs):
        batch = all_batches[lab]
        available, values = profiles_1min[lab]
        start = batch.time[0]
        end = ctx['v'].horizons[lab] if lab in historical else batch.time[-1]
        wait = next(row['Prefijo original [min]'] for row in coverage if row['LAB'] == lab)
        original_times = np.r_[batch.time[batch.time < end], end]
        ax.plot(original_times, np.interp(original_times, batch.time, batch.temperature_c),
                color='#0072B2', label='Temperatura actual', lw=1.5)
        mask = (available > start) & (available < end)
        times = np.r_[start, available[mask], end]
        temperatures = np.r_[causal_temperature(batch, start), values[mask], causal_temperature(batch, end)]
        if wait > 0:
            ax.axvspan(start, start + wait/60, color='gray', alpha=.2, label='Prefijo original')
            times, temperatures = times[1:], temperatures[1:]
        ax.step(times, temperatures, where='post', color='#D55E00', lw=.9, label='Mediana causal')
        ax.set(title=f'{lab} · {roles[lab]}', xlabel='Tiempo [h]', ylabel='Temperatura [°C]')
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

temperature_figure(list(calibration), 'Temperatura · calibración química')
temperature_figure([*historical, *recent], 'Temperatura · calibración de gas y holdout')

In [ ]:
# Se limita el paso de LSODA en la variante causal para resolver los escalones de un minuto.
# El perfil actual se ejecuta con el paso y las tolerancias del notebook de origen.
def causal_solver(solver, step):
    def solve(rhs, interval, initial, **kwargs):
        return solver(rhs, interval, initial, **{**kwargs, 'max_step': step})
    return solve

def predict(context, model, causal=False, step=1/60):
    is_single = model == 'S'
    simulate = context['simulate_s'] if is_single else context['simulate']
    parameters = context['ps'] if is_single else context['theta']
    curve_for_plot = context['plotframe_s'] if is_single else context['with_jumps']
    scoped = context.copy()
    if causal:
        scoped['solve_ivp'] = causal_solver(context['solve_ivp'], step)
    simulate = FunctionType(simulate.__code__, scoped)
    # La sustitución temporal se aplica a cinética, fuente de gas y solubilidad.
    temperature = causal_temperature if causal else reference_temperature
    with patch.object(kin, 'temperature_at', temperature):
        frames = {lab: simulate(batch, parameters)
                  for lab, batch in {**calibration, **historical}.items()}
        scoped.update(batches=context['batches'], caches={}, activation={})
        sugar = 'S' if is_single else 'GF'
        for lab, batch in historical.items():
            frame = curve_for_plot(frames[lab])
            end = frame.index[-1]
            times = np.r_[np.arange(0., end, .1), end]
            states = context['STATES'] if is_single else kin.STATE_NAMES
            trajectory = np.column_stack([np.interp(times, frame.index, frame[s]) for s in states])
            rates = []
            for t, y in zip(times, trajectory):
                if is_single:
                    mu, beta, _, _ = scoped['terms_s'](t, y, parameters, batch)
                else:
                    terms = kin.kinetic_terms(parameters, temperature(batch, t), *y)
                    mu = terms['mu'] if y[2] > 0 and y[3] > 0 else 0.
                    beta = terms['beta_g'] + terms['beta_f']
                rates.append((mu / 12.72 + beta / 1.05) * y[0])
            scoped['caches'][lab] = (times, np.asarray(rates))
            consumed = batch.initials['G'] + batch.initials['F'] - np.interp(times, frame.index, frame[sugar])
            z = np.clip((consumed - 5.) / 5., 0., 1.)
            scoped['activation'][lab] = z * z * (3. - 2. * z)
        FunctionType(context['configure_gas'].__code__, scoped, argdefs=(False,))(
            {lab: frames[lab] for lab in historical}, is_single)
        gas_function = FunctionType(context['gas'].__code__, scoped, argdefs=context['gas'].__defaults__)
        gas_frames = {lab: gas_function(lab, *context['gas_parameters']) for lab in historical}

        # Las definiciones mantienen el manejo del agotamiento propio del holdout LAB022–024.
        scoped['simulate'] = context['simulate']
        run(scoped, 'def simulate_holdout' + cell(single if is_single else gf, 'def simulate_holdout')
            .split('def simulate_holdout', 1)[1].split('\nholdout_kinetic,')[0])
        run(scoped, context['predict_holdout'])
    return SimpleNamespace(frames={**frames, **scoped['holdout_kinetic']},
                           plot={**{lab: curve_for_plot(f) for lab, f in frames.items()},
                                 **scoped['holdout_plot']},
                           gas={**gas_frames, **scoped['holdout_gas']},
                           mean_gas=scoped['holdout_mean_gas'])

results = {}
for model, context in contexts.items():
    results[model] = (predict(context, model), predict(context, model, causal=True))
    print(f'{model}: {len(results[model][0].frames)} lotes por variante')

In [ ]:
labels = {
    'X': ('Biomasa', 'g/L'), 'N': ('YAN', 'mg N/L'),
    'G': ('Glucosa', 'g/L'), 'F': ('Fructosa', 'g/L'), 'GF': ('Azúcar total', 'g/L'),
    'E': ('Etanol', 'g/L'), 'CO2': ('Flujo de CO₂', 'g/(L h)'),
}
recent_columns = {'X': 'X_g_L', 'N': 'YAN for plot', 'G': 'GLUCOSE-320',
                  'F': 'Fructose (derived)', 'GF': 'GLU-FRU-320'}

def samples(lab, variable):
    if variable == 'CO2':
        if lab in recent:
            y = ctx['holdout_flow'][lab].dropna()
            return y.index.to_numpy(), y.to_numpy(), 'Sensor · 30 min'
        o = ctx['obs'][lab]
        return o.time_h.to_numpy(), o.co2_raw_g_l_h.to_numpy(), 'Sensor · 1 h'
    if lab in calibration:
        b = calibration[lab]
        y = b.observations[variable] if variable != 'GF' else b.observations['G'] + b.observations['F']
        return b.time, y * (1000 if variable == 'N' else 1), 'Muestras tratadas'
    if lab in recent:
        if variable not in recent_columns:
            return np.array([]), np.array([]), ''
        table = ctx['holdout_biomass'] if variable == 'X' else ctx['holdout_chemistry']
        z = table.loc[table.lab.eq(lab)].dropna(subset=[recent_columns[variable]])
        return z.hours.to_numpy(), z[recent_columns[variable]].to_numpy(), 'Muestras'
    v = ctx['v']
    if lab in ctx['TRAIN']:
        z = ctx['chemical_observations'](v.chem, lab, variable)
        return z.t_h.to_numpy(), z[ctx['CHEMICAL_COLUMNS'][variable]].to_numpy() * (1000 if variable == 'N' else 1), 'Muestras'
    if variable == 'X':
        z = v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h', 'X_viable_g_l'])
        return z.t_h.to_numpy(), z.X_viable_g_l.to_numpy(), 'Muestras'
    if variable == 'GF':
        z = v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h', 'density_g_l'])
        return z.t_h.to_numpy(), 2.5 * (z.density_g_l.to_numpy() - 1000), 'Azúcar por densidad'
    return np.array([]), np.array([]), ''

def prediction(result, model, lab, variable, for_plot):
    if variable == 'CO2':
        return result.mean_gas[lab] if lab in recent else result.gas[lab]['qpred']
    state = 'S' if variable == 'GF' and model == 'S' else variable
    curves = result.plot if for_plot else result.frames
    # En la calibración de gas se conserva el tratamiento del salto del notebook original.
    if not for_plot and lab in historical:
        curves = result.plot
    return curves[lab][state] * (1000 if variable == 'N' else 1)

def evaluate(model, lab, variable):
    batch = all_batches[lab]
    start = batch.time[0]
    end = ctx['v'].horizons[lab] if lab in historical else batch.time[-1]
    t, y, origin = samples(lab, variable)
    valid = np.isfinite(y) & (t <= end)
    score = valid & (t >= start if variable == 'CO2' else t > start + 1e-9)
    curves = [prediction(result, model, lab, variable, False) for result in results[model]]
    for curve in curves:
        score &= (t >= curve.index[0] - 1e-9) & (t <= curve.index[-1] + 1e-9)
    rmse = [float(np.sqrt(np.mean((np.interp(t[score], curve.index, curve) - y[score]) ** 2)))
            if score.any() else np.nan for curve in curves]
    return dict(Modelo=model, LAB=lab, Rol=roles[lab], Variable=variable, Referencia=origin,
                Unidad=labels[variable][1], n=int(score.sum()), Actual=rmse[0], Causal=rmse[1],
                Cambio=rmse[1] - rmse[0])

report = {}
for model in results:
    for lab in all_batches:
        for variable in labels:
            if (model == 'S' and variable in ('G', 'F')) or (variable == 'CO2' and lab in calibration):
                continue
            report[model, lab, variable] = evaluate(model, lab, variable)
metrics = pd.DataFrame(report.values())
for model in results:
    table = metrics.loc[metrics.Modelo.eq(model)]
    for variables in [('X', 'N'), ('G', 'F', 'GF', 'E', 'CO2')]:
        display(table.loc[table.Variable.isin(variables)].drop(columns='Modelo').round(4))

In [ ]:
def compare_figure(model, variable, labs, group):
    rows = len(labs) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(15, 3.5 * rows))
    label, unit = labels[variable]
    for ax, lab in zip(axes.flat, labs):
        batch = all_batches[lab]
        end = ctx['v'].horizons[lab] if lab in historical else batch.time[-1]
        t, y, origin = samples(lab, variable)
        visible = np.isfinite(y) & (t <= end)
        if visible.any():
            if variable == 'CO2':
                ax.plot(t[visible], y[visible], color='black', lw=.8, label=origin)
            else:
                marker = 'D' if origin == 'Azúcar por densidad' else 'o'
                ax.scatter(t[visible], y[visible], color='black', marker=marker, s=16, label=origin, zorder=5)
        for result, name, color, style in zip(results[model], ['Actual', 'Causal 1 min'],
                                              ['#0072B2', '#D55E00'], ['-', '--']):
            curve = prediction(result, model, lab, variable, True)
            curve = curve.loc[curve.index <= end]
            ax.plot(curve.index, curve, label=name, color=color, ls=style, lw=1.4)
        if variable == 'N':
            for i, (t_pulse, _) in enumerate(batch.pulses['N']):
                ax.axvline(t_pulse, c='gray', ls=':', lw=.8, label='Pulso N' if i == 0 else None)
        row = report[model, lab, variable]
        metric = (f"RMSE actual: {row['Actual']:.3f}\nRMSE causal: {row['Causal']:.3f}"
                  if row['n'] else 'Sin muestras para RMSE')
        ax.legend(title=metric, fontsize=7, title_fontsize=7)
        ax.set(title=f'{lab} · {roles[lab]}', xlabel='Tiempo [h]', ylabel=f'{label} [{unit}]')
        ax.grid(alpha=.2)
        if lab in recent:
            ax.set(xlim=(-2, 192), xticks=[0, 25, 50, 75, 100, 125, 150, 175, 190])
    fig.suptitle(f'{model} · {label} · {group}')
    fig.tight_layout()
    plt.show()

## G/F · curvas por lote

In [ ]:
compare_figure('G/F', 'X', list(calibration), 'calibración química')
compare_figure('G/F', 'X', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('G/F', 'N', list(calibration), 'calibración química')
compare_figure('G/F', 'N', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('G/F', 'G', list(calibration), 'calibración química')
compare_figure('G/F', 'G', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('G/F', 'F', list(calibration), 'calibración química')
compare_figure('G/F', 'F', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('G/F', 'GF', list(calibration), 'calibración química')
compare_figure('G/F', 'GF', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('G/F', 'E', list(calibration), 'calibración química')
compare_figure('G/F', 'E', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('G/F', 'CO2', [*historical, *recent], 'calibración de gas y holdout')

## S · curvas por lote

In [ ]:
compare_figure('S', 'X', list(calibration), 'calibración química')
compare_figure('S', 'X', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('S', 'N', list(calibration), 'calibración química')
compare_figure('S', 'N', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('S', 'GF', list(calibration), 'calibración química')
compare_figure('S', 'GF', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('S', 'E', list(calibration), 'calibración química')
compare_figure('S', 'E', [*historical, *recent], 'calibración de gas y holdout')

In [ ]:
compare_figure('S', 'CO2', [*historical, *recent], 'calibración de gas y holdout')

## Lectura del cambio de RMSE

El cambio es RMSE causal menos RMSE actual. Un valor positivo significa más error frente a las mismas muestras. Los RMSE de LAB004–009 describen la sensibilidad en los lotes que calibraron la cinética. Los cambios de YAN en LAB016–018 se pueden observar en las curvas. Esos lotes carecen de muestras posteriores a la inicial para puntuar YAN con el soporte de los notebooks 16 y 17.

In [ ]:
key = metrics.loc[metrics.Variable.isin(['X', 'N']) & metrics.n.gt(0)].copy()
key['Magnitud del cambio'] = key.Cambio.abs()
display(key.sort_values('Magnitud del cambio', ascending=False).head(12).round(4))
print('RESUMEN_X_N', key[['Modelo', 'LAB', 'Variable', 'Actual', 'Causal', 'Cambio']].round(5).to_json(orient='records'))